# Silver Layer — Facts (`dim_date` + `fact_transactions`)

**Scop:** Construirea tabelei centrale de fapte și a dimensiunii de timp.

**Componente:**
1. **`dim_date`** — calendar static, 2020-2030 (~4000 rânduri)
2. **`fact_transactions`** — ~50.000 tranzacții cu validări complexe, deduplicare, lookup SCD2 pentru `customer_sk`

**Validări critice pentru `fact_transactions`:**
- `amount > 0` (prinde erorile cu sumă negativă/null)
- `status_code` valid (FK către `ref_transaction_statuses`)
- `channel_code` valid (FK către `ref_channels`)
- `type_code` valid (FK către `ref_transaction_types`)
- `account_id` valid (FK către `dim_accounts`)
- `card_id` valid când populat (FK către `dim_cards`)
- `completed_at >= initiated_at` (prinde inversarea logică)
- `amount_ron ≈ amount * exchange_rate` cu toleranță 0.01 RON

**Deduplicare:** pe `transaction_id` păstrăm rândul cu `source_ingestion_ts` cel mai recent.

**Lookup SCD Type 2:** pentru fiecare tranzacție, găsim versiunea `dim_customers` validă la momentul `initiated_at`. Funcționează cu convenția high-date sentinel `valid_to = '3000-01-01'` (vezi 02b2).

**Ordine în pipeline:** Acest notebook face `OVERWRITE` pe `fact_transactions` și pe `qrt_transactions`. Notebook-ul `02d_silver_corrections` rulează **după** și completează `fact_transactions` cu tranzacțiile auto-fixate din quarantine. Ordinea canonică este `02c → 02d` — re-rularea 02c fără 02d după duce la pierderea corecțiilor (vezi orchestrarea recomandată în Databricks Workflow).

## 1. Configurare

In [0]:
from datetime import datetime, date
from pyspark.sql import DataFrame, Window
from pyspark.sql.functions import (
    col, when, lit, current_timestamp, to_date, to_timestamp,
    array, size, concat_ws, abs as spark_abs,
    row_number, sequence, explode, dayofweek, dayofmonth,
    month, year, weekofyear, quarter, last_day, date_format,
    expr, broadcast,
    filter as array_filter,
)
from pyspark.sql.types import IntegerType, DoubleType, StringType, DateType

CATALOG     = "banking"
BRONZE      = "bronze"
SILVER      = "silver"
QUARANTINE  = "silver_quarantine"

DATE_DIM_START = "2020-01-01"
DATE_DIM_END   = "2030-12-31"

print(f"Bronze:     {CATALOG}.{BRONZE}.*")
print(f"Silver:     {CATALOG}.{SILVER}.*")
print(f"Quarantine: {CATALOG}.{QUARANTINE}.*")
print(f"dim_date:   {DATE_DIM_START} ... {DATE_DIM_END}")

Bronze:     banking.bronze.*
Silver:     banking.silver.*
Quarantine: banking.silver_quarantine.*
dim_date:   2020-01-01 ... 2030-12-31


## 2. Funcție generică de validare (reutilizată din 02b1)

In [0]:
def split_valid_invalid(
    df: DataFrame, 
    validations: list
) -> tuple:
    """
    Aplica o lista de validari pe DataFrame.
    
    validations: lista de tupluri (rule_name, condition_expression)
        - condition_expression = TRUE inseamna 'rand valid'
    
    Returneaza (df_valid, df_invalid).
    df_invalid contine coloana extra 'error_reasons' (array de strings).
    """
    error_cols = []
    for rule_name, condition in validations:
        col_name = f"_err_{rule_name}"
        df = df.withColumn(col_name, when(~condition, lit(rule_name)))
        error_cols.append(col_name)
    
    df = df.withColumn(
        "error_reasons",
        array_filter(array(*[col(c) for c in error_cols]), lambda x: x.isNotNull())
    )
    df = df.drop(*error_cols)
    
    df_valid   = df.filter(size(col("error_reasons")) == 0).drop("error_reasons")
    df_invalid = df.filter(size(col("error_reasons")) > 0)
    
    return df_valid, df_invalid

## 3. `dim_date` — calendar static

Generăm un rând pentru fiecare zi din intervalul 2020-2030 cu atribute calendaristice.

**Coloane:**
- `date_id` — INT cu format YYYYMMDD (ex: 20260415) — surogate key natural
- `full_date` — DATE
- `year`, `quarter`, `month`, `month_name`, `week`, `day`, `day_of_week`, `day_name`
- `is_weekend` — flag boolean
- `is_month_end` — flag pentru rapoarte de sfârșit de lună
- `fiscal_quarter` — Q1-Q4 cu prefix anul

**Tehnică:** folosim `sequence()` pentru a genera array-ul de date, apoi `explode()`.

In [0]:
print(f"Generare dim_date: {DATE_DIM_START} ... {DATE_DIM_END}")

# Generam un singur rand cu un array de date, apoi il "explodam"
df_dates = (
    spark.range(1)
    .select(
        explode(
            sequence(
                to_date(lit(DATE_DIM_START)),
                to_date(lit(DATE_DIM_END)),
                expr("interval 1 day")
            )
        ).alias("full_date")
    )
)

# Adaugam atributele calendaristice
df_dates = (df_dates
    .withColumn("date_id",         (year("full_date") * 10000 + month("full_date") * 100 + dayofmonth("full_date")).cast(IntegerType()))
    .withColumn("year",            year("full_date"))
    .withColumn("quarter",         quarter("full_date"))
    .withColumn("month",           month("full_date"))
    .withColumn("month_name",      date_format("full_date", "MMMM"))
    .withColumn("week",            weekofyear("full_date"))
    .withColumn("day",             dayofmonth("full_date"))
    .withColumn("day_of_week",     dayofweek("full_date"))      # 1=Sunday, 7=Saturday
    .withColumn("day_name",        date_format("full_date", "EEEE"))
    .withColumn("is_weekend",      dayofweek("full_date").isin([1, 7]))   # duminica sau sambata
    .withColumn("is_month_end",    col("full_date") == last_day("full_date"))
    .withColumn("fiscal_quarter",  concat_ws("-", year("full_date"), concat_ws("", lit("Q"), quarter("full_date"))))
    .withColumn("silver_loaded_at", current_timestamp())
)

# Reordonam coloanele cu date_id primul
target_order = [
    "date_id", "full_date", "year", "quarter", "month", "month_name",
    "week", "day", "day_of_week", "day_name",
    "is_weekend", "is_month_end", "fiscal_quarter", "silver_loaded_at"
]
df_dates = df_dates.select(*target_order)

# Scriem in Silver
(df_dates.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.dim_date"))

print(f"  dim_date creata: {df_dates.count()} randuri")

Generare dim_date: 2020-01-01 ... 2030-12-31


In [0]:
%sql
-- Verificare dim_date — sample
SELECT * FROM banking.silver.dim_date
WHERE full_date IN ('2026-01-01', '2026-04-15', '2026-12-31')
ORDER BY full_date;

date_id,full_date,year,quarter,month,month_name,week,day,day_of_week,day_name,is_weekend,is_month_end,fiscal_quarter,silver_loaded_at
20260101,2026-01-01,2026,1,1,January,1,1,5,Thursday,false,false,2026-Q1,2026-05-01T20:24:39.888Z
20260415,2026-04-15,2026,2,4,April,16,15,4,Wednesday,false,false,2026-Q2,2026-05-01T20:24:39.888Z
20261231,2026-12-31,2026,4,12,December,53,31,5,Thursday,false,true,2026-Q4,2026-05-01T20:24:39.888Z


## 4. `fact_transactions` — citire Bronze și pregătire

In [0]:
df = spark.table(f"{CATALOG}.{BRONZE}.raw_transactions")

# Lineage metadata
if "_batch_id" in df.columns:
    df = df.withColumnRenamed("_batch_id", "source_batch_id")
if "_ingestion_ts" in df.columns:
    df = df.withColumnRenamed("_ingestion_ts", "source_ingestion_ts")
for c in ["_source_system", "_source_file"]:
    if c in df.columns:
        df = df.drop(c)

# Type casting
df = df.withColumn("amount",         col("amount").cast(DoubleType()))
df = df.withColumn("amount_ron",     col("amount_ron").cast(DoubleType()))
df = df.withColumn("exchange_rate",  col("exchange_rate").cast(DoubleType()))
df = df.withColumn("initiated_at",   to_timestamp("initiated_at"))
df = df.withColumn("completed_at",   to_timestamp("completed_at"))

print(f"Bronze rows: {df.count()}")
print(f"Coloane:")
df.printSchema()

Bronze rows: 50252
Coloane:
root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- card_id: string (nullable = true)
 |-- counterpart_account: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- type_code: string (nullable = true)
 |-- status_code: string (nullable = true)
 |-- channel_code: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- amount_ron: double (nullable = true)
 |-- exchange_rate: double (nullable = true)
 |-- description: string (nullable = true)
 |-- reference_code: string (nullable = true)
 |-- initiated_at: timestamp (nullable = true)
 |-- completed_at: timestamp (nullable = true)
 |-- created_at: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- source_ingestion_ts: timestamp (nullable = true)
 |-- source_batch_id: string (nullable = true)



## 5. Deduplicare pe `transaction_id`

În Bronze pot exista duplicate (injectate intenționat). Păstrăm pentru fiecare `transaction_id` rândul cu `source_ingestion_ts` cel mai recent.

In [0]:
n_before = df.count()
n_distinct = df.select("transaction_id").distinct().count()
n_duplicates = n_before - n_distinct

print(f"Inainte:    {n_before:>6d} randuri")
print(f"Distinct:   {n_distinct:>6d} transaction_id unici")
print(f"Duplicate:  {n_duplicates:>6d} ({n_duplicates/n_before*100:.1f}%)")

if n_duplicates > 0:
    # Window: pentru fiecare transaction_id, ordoneaza dupa source_ingestion_ts DESC
    # Pastram doar randul cu rownum=1 (cel mai recent)
    window = Window.partitionBy("transaction_id").orderBy(col("source_ingestion_ts").desc())
    df = df.withColumn("_rn", row_number().over(window)).filter(col("_rn") == 1).drop("_rn")
    print(f"Dupa dedup: {df.count():>6d} randuri")
else:
    print("Nu sunt duplicate de eliminat")

Inainte:     50252 randuri
Distinct:    50252 transaction_id unici
Duplicate:       0 (0.0%)
Nu sunt duplicate de eliminat


## 6. Pregătire FK references (broadcast pentru performanță)

In [0]:
# Coduri valide din nomenclatoare Silver
valid_status_codes  = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_transaction_statuses")
                       .select("status_code").collect()]
valid_channel_codes = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_channels")
                       .select("channel_code").collect()]
valid_type_codes    = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_transaction_types")
                       .select("type_code").collect()]
valid_currencies    = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_currencies")
                       .select("currency_code").collect()]

# FK pe dimensiuni
valid_account_ids = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_accounts")
                     .select("account_id").collect()]
valid_card_ids    = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_cards")
                     .select("card_id").collect()]
valid_employee_ids = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_employees")
                      .select("employee_id").collect()]

print(f"Coduri valide:")
print(f"  Status:       {len(valid_status_codes)}")
print(f"  Channel:      {len(valid_channel_codes)}")
print(f"  Type:         {len(valid_type_codes)}")
print(f"  Currency:     {len(valid_currencies)}")
print(f"FK dimensiuni:")
print(f"  Accounts:     {len(valid_account_ids)}")
print(f"  Cards:        {len(valid_card_ids)}")
print(f"  Employees:    {len(valid_employee_ids)}")

Coduri valide:
  Status:       5
  Channel:      7
  Type:         13
  Currency:     6
FK dimensiuni:
  Accounts:     800
  Cards:        586
  Employees:    60


## 7. Validări `fact_transactions`

Aplicăm 12 reguli de calitate. Înregistrările care încalcă măcar una merg în quarantine.

In [0]:
# Toleranta pentru consistenta amount_ron (0.05 RON pentru a acomoda rotunjiri)
TOL = 0.05

validations = [
    # PK si NOT NULL
    ("pk_null",              col("transaction_id").isNotNull()),
    ("account_id_null",      col("account_id").isNotNull()),
    
    # Validari numerice
    ("amount_null",          col("amount").isNotNull()),
    ("amount_positive",      (col("amount").isNull()) | (col("amount") > 0)),
    
    # FK validari
    ("account_id_invalid",   col("account_id").isin(valid_account_ids)),
    ("status_invalid",       col("status_code").isin(valid_status_codes)),
    ("channel_invalid",      col("channel_code").isin(valid_channel_codes)),
    ("type_invalid",         col("type_code").isin(valid_type_codes)),
    ("currency_invalid",     col("currency_code").isin(valid_currencies)),
    
    # FK opționale (când sunt populate, trebuie să fie valide)
    ("card_id_invalid", 
        (col("card_id").isNull()) | (col("card_id").isin(valid_card_ids))),
    ("employee_id_invalid", 
        (col("employee_id").isNull()) | (col("employee_id").isin(valid_employee_ids))),
    
    # Consistenta logica temporala
    ("completed_before_initiated", 
        (col("completed_at").isNull()) | (col("completed_at") >= col("initiated_at"))),
    
    # Consistenta logica numerica: amount_ron ≈ amount * exchange_rate
    ("amount_ron_inconsistent",
        (col("amount").isNull()) | (col("amount_ron").isNull()) | (col("exchange_rate").isNull()) |
        (spark_abs(col("amount_ron") - col("amount") * col("exchange_rate")) <= lit(TOL))),
]

df_valid, df_invalid = split_valid_invalid(df, validations)

n_valid   = df_valid.count()
n_invalid = df_invalid.count()
total     = n_valid + n_invalid
pct       = (n_invalid / total * 100) if total > 0 else 0

print(f"Valid:   {n_valid:>6d}")
print(f"Invalid: {n_invalid:>6d}  ({pct:.1f}% quarantine)")

# Scriem quarantine
if n_invalid > 0:
    (df_invalid
        .withColumn("quarantine_loaded_at", current_timestamp())
        .withColumn("error_reasons_str", concat_ws(", ", col("error_reasons")))
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{QUARANTINE}.qrt_transactions"))
    print(f"  Quarantine scris in qrt_transactions")

Valid:    46642
Invalid:   3610  (7.2% quarantine)
  Quarantine scris in qrt_transactions


## 8. Lookup SCD Type 2 — `customer_sk` la momentul tranzacției

**Pasul cel mai interesant.** Pentru fiecare tranzacție trebuie să găsim:
1. Cine e clientul? — prin JOIN `account_id → customer_id`
2. Care era versiunea SCD2 a clientului la momentul `initiated_at`? — prin JOIN cu `dim_customers` cu condiție temporală

**Logica SCD2 lookup:**
- O tranzacție din 15 martie pe un client care era RETAIL atunci → ia `customer_sk` al versiunii RETAIL
- O tranzacție din 15 mai pe același client (devenit PREMIUM pe 1 aprilie) → ia `customer_sk` al versiunii PREMIUM

**Condiție de match:**
```
dim.customer_id = account.customer_id
AND dim.valid_from <= tx.initiated_at
AND (dim.valid_to > tx.initiated_at OR dim.valid_to IS NULL)
```

In [0]:
# Pasul 1: JOIN cu dim_accounts pentru a obtine customer_id
df_with_cust = df_valid.alias("tx").join(
    spark.table(f"{CATALOG}.{SILVER}.dim_accounts").select("account_id", "customer_id").alias("acc"),
    col("tx.account_id") == col("acc.account_id"),
    "left"
).select("tx.*", col("acc.customer_id").alias("customer_id"))

print(f"Dupa JOIN cu dim_accounts: {df_with_cust.count()} randuri")

# Pasul 2: JOIN cu dim_customers (SCD2) pentru a obtine customer_sk valid la initiated_at
dim_customers = spark.table(f"{CATALOG}.{SILVER}.dim_customers").select(
    col("customer_id").alias("dim_customer_id"),
    col("customer_sk"),
    col("valid_from"),
    col("valid_to"),
    col("is_current"),
)

df_with_sk = df_with_cust.alias("tx").join(
    dim_customers.alias("dim"),
    (col("tx.customer_id") == col("dim.dim_customer_id")) &
    (col("dim.valid_from") <= col("tx.initiated_at")) &
    (col("dim.valid_to") > col("tx.initiated_at")),
    "left"
).select(
    "tx.*",
    col("dim.customer_sk").alias("customer_sk")
)

# Verificare: cate tranzactii nu au gasit customer_sk?
n_no_sk = df_with_sk.filter(col("customer_sk").isNull()).count()
print(f"Tranzactii fara customer_sk lookup: {n_no_sk}")

if n_no_sk > 0:
    print("  ATENTIE: Aceste tranzactii vor avea customer_sk=NULL.")
    print("  Cauze posibile: customer-ul nu exista in dim_customers (a fost in quarantine)")

Dupa JOIN cu dim_accounts: 46642 randuri
Tranzactii fara customer_sk lookup: 0


## 9. Lookup `date_id` din `dim_date`

Adăugăm `date_id` (format YYYYMMDD INT) pentru fiecare tranzacție bazat pe data lui `initiated_at`. Asta ne permite JOIN-uri rapide cu `dim_date` în Gold pentru group by pe perioade.

In [0]:
df_with_sk = df_with_sk.withColumn(
    "date_id",
    (year("initiated_at") * 10000 + month("initiated_at") * 100 + dayofmonth("initiated_at")).cast(IntegerType())
)

print("date_id calculat din initiated_at")

date_id calculat din initiated_at


## 10. Selectare coloane finale și scriere

In [ ]:
# Definim ordinea finala a coloanelor pentru fact_transactions
final_columns = [
    # Cheia primara
    "transaction_id",
    
    # Foreign keys catre dimensiuni
    "customer_sk",          # SCD2 surrogate
    "customer_id",          # natural key (pentru filtre rapide)
    "account_id",
    "card_id",
    "employee_id",
    "date_id",              # FK catre dim_date
    
    # Atribute clasificare
    "type_code",
    "status_code",
    "channel_code",
    "currency_code",
    
    # Masuri (numerical facts)
    "amount",
    "amount_ron",
    "exchange_rate",
    
    # Atribute descriptive
    "counterpart_account",
    "description",
    "reference_code",
    
    # Timestamp-uri
    "initiated_at",
    "completed_at",
    
    # Lineage si audit
    "source_batch_id",
    "source_ingestion_ts",
]

# Construim DataFrame-ul final
df_final = df_with_sk.select(*[col(c) for c in final_columns if c in df_with_sk.columns])
df_final = df_final.withColumn("silver_loaded_at", current_timestamp())

print(f"Coloane fact_transactions: {len(df_final.columns)}")
print(f"Schema finala:")
df_final.printSchema()

# Scriem in Silver
(df_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.fact_transactions"))

print(f"\nfact_transactions scris in Silver: {df_final.count()} randuri")

## 11. Verificare — sumar fact_transactions

In [0]:
%sql
SELECT 
    COUNT(*)                                    AS total_transactions,
    COUNT(DISTINCT transaction_id)              AS unique_tx_ids,
    COUNT(DISTINCT account_id)                  AS unique_accounts,
    COUNT(DISTINCT customer_id)                 AS unique_customers,
    SUM(CASE WHEN customer_sk IS NULL THEN 1 ELSE 0 END) AS missing_customer_sk,
    SUM(CASE WHEN card_id IS NOT NULL THEN 1 ELSE 0 END) AS with_card,
    SUM(CASE WHEN employee_id IS NOT NULL THEN 1 ELSE 0 END) AS with_employee,
    ROUND(SUM(amount_ron), 2)                   AS total_volume_ron,
    ROUND(AVG(amount_ron), 2)                   AS avg_amount_ron,
    MIN(initiated_at)                           AS first_tx,
    MAX(initiated_at)                           AS last_tx
FROM banking.silver.fact_transactions;

total_transactions,unique_tx_ids,unique_accounts,unique_customers,missing_customer_sk,with_card,with_employee,total_volume_ron,avg_amount_ron,first_tx,last_tx
46642,46642,800,405,0,20708,1867,6.058936244E7,1299.03,2025-05-01T00:34:30.907Z,2026-05-01T23:22:43.907Z


## 12. Distribuții și sanity checks

In [0]:
%sql
-- Distributie pe status (verificam ca nu avem statusuri invalide)
SELECT 
    status_code,
    COUNT(*) AS n_tx,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM banking.silver.fact_transactions
GROUP BY status_code
ORDER BY n_tx DESC;

status_code,n_tx,pct
COMPLETED,40806,87.5
PENDING,2422,5.2
FAILED,2006,4.3
REVERSED,920,2.0
ON_HOLD,488,1.0


In [0]:
%sql
-- Distributie pe canal
SELECT 
    channel_code,
    COUNT(*) AS n_tx,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(SUM(amount_ron), 2) AS total_volume_ron
FROM banking.silver.fact_transactions
GROUP BY channel_code
ORDER BY n_tx DESC;

channel_code,n_tx,pct,total_volume_ron
POS,17707,38.0,2.423430957E7
ONLINE,11667,25.0,1.474834172E7
ATM,8389,18.0,1.017794415E7
MOBILE,5569,11.9,7138389.88
BRANCH,2402,5.1,2840898.99
DIRECT_DEBIT,908,1.9,1449478.13


In [0]:
%sql
-- Distributie pe valuta
SELECT 
    currency_code,
    COUNT(*) AS n_tx,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct,
    ROUND(AVG(amount), 2) AS avg_amount,
    ROUND(AVG(amount_ron), 2) AS avg_amount_ron
FROM banking.silver.fact_transactions
GROUP BY currency_code
ORDER BY n_tx DESC;

currency_code,n_tx,pct,avg_amount,avg_amount_ron
RON,38224,81.95,767.16,767.16
EUR,5637,12.09,758.11,3767.78
USD,1858,3.98,718.28,3311.26
GBP,467,1.00,792.61,4620.9
CHF,456,0.98,738.01,3763.83


## 13. Inspecție quarantine — exemple de erori detectate

In [0]:
%sql
-- Exemple de erori (se vede tipul fiecarei erori)
SELECT 
    transaction_id,
    amount,
    status_code,
    channel_code,
    amount_ron,
    exchange_rate,
    error_reasons_str
FROM banking.silver_quarantine.qrt_transactions
LIMIT 20;

transaction_id,amount,status_code,channel_code,amount_ron,exchange_rate,error_reasons_str
TXN-00000008,711.54,COMPLETED,ATM,711.54,1.0,completed_before_initiated
TXN-00000013,248.28,COMPLETED,ATM,9185.31,4.97,amount_ron_inconsistent
TXN-00000038,35.18,COMPLETED,INVALID_CHANNEL,174.84,4.97,channel_invalid
TXN-00000055,692.49,COMPLETED,ONLINE,4872.94,1.0,amount_ron_inconsistent
TXN-00000064,93.98,COMPLETED,ONLINE,93.98,1.0,"card_id_invalid, completed_before_initiated"
TXN-00000074,1385.81,COMPLETED,INVALID_CHANNEL,6887.48,4.97,channel_invalid
TXN-00000079,263.33,COMPLETED,ONLINE,1405.38,1.0,amount_ron_inconsistent
TXN-00000090,353.28,PENDING,INVALID_CHANNEL,353.28,1.0,channel_invalid
TXN-00000147,33.64,COMPLETED,BRANCH,285.19,1.0,amount_ron_inconsistent
TXN-00000148,88.19,COMPLETED,BRANCH,363.51,1.0,amount_ron_inconsistent


In [0]:
%sql
-- Statistici per tip de eroare
SELECT 
    reason,
    COUNT(*) AS n_tx
FROM banking.silver_quarantine.qrt_transactions
LATERAL VIEW explode(error_reasons) AS reason
GROUP BY reason
ORDER BY n_tx DESC;

reason,n_tx
amount_ron_inconsistent,1693
completed_before_initiated,641
card_id_invalid,546
channel_invalid,401
status_invalid,362


## 14. Test SCD Type 2 — ce se întâmplă cu tranzacțiile clienților modificați

Dacă un client a fost modificat (segment, kyc, county), tranzacțiile sale sunt împărțite pe `customer_sk` diferiți în funcție de momentul tranzacției.

In [0]:
%sql
-- Clientii care au mai mult de 1 customer_sk in fact_transactions
-- (au fost modificati intre timp)
SELECT 
    ft.customer_id,
    COUNT(DISTINCT ft.customer_sk) AS n_versions_used,
    COUNT(*)                       AS n_transactions
FROM banking.silver.fact_transactions ft
WHERE ft.customer_sk IS NOT NULL
GROUP BY ft.customer_id
HAVING COUNT(DISTINCT ft.customer_sk) > 1
ORDER BY n_transactions DESC
LIMIT 10;

customer_id,n_versions_used,n_transactions


## Sumar 02c — Silver Facts

Realizat:
- **`dim_date`** generată static (~4000 rânduri, 2020-2030) cu atribute calendaristice
- **Deduplicare** pe `transaction_id` (păstrăm ingestia cea mai recentă)
- **12 reguli de validare** complete pentru `fact_transactions`
- **SCD Type 2 lookup** pentru `customer_sk` corect la momentul tranzacției
- **Lookup `date_id`** din `dim_date` (format YYYYMMDD INT)
- **Quarantine** cu `error_reasons` per regulă pentru audit
- **Lineage** păstrat (`source_batch_id`, `source_ingestion_ts`, `silver_loaded_at`)

**Următorul pas — Silver complete:**
- `02d_silver_corrections` — auto-fix + manual override pentru quarantine

**Apoi:** Gold layer — `kpi_transactions_daily`, `customer_segments_rfm`, `report_pnl_monthly`, `fraud_features`.